# Day 22: Deep Learning Mini-Project & Checkpoint

Today **you** build and train a small neural network, alone, from start to end.

**Checkpoint:** at the end of today you can say:
> "I understand what this code is doing, and I can train a small neural network."

| Part | What you do | Time |
|---|---|---|
| A | Read code and explain each line | 25 min |
| B | Mini-project: predict exam results | 75 min |
| C | Experiments: change settings, compare | 25 min |
| D | Answer 6 short questions | 15 min |
| ✅ | Checkpoint list | 5 min |

**Rules:** You can look at your old notebooks. Ask the teacher only after you try for 5 minutes.

In [1]:
import numpy as np                                     # arrays of numbers
import pandas as pd                                    # tables (read CSV files)
import matplotlib.pyplot as plt                        # charts
import torch                                           # PyTorch: tensors + training
from torch import nn                                   # layers, ReLU, loss functions
from sklearn.model_selection import train_test_split   # split into train / validation / test
from sklearn.preprocessing import StandardScaler       # make columns the same scale

---
# Part A: Read the code and explain it

This code trains a small network on **loans.csv**: will the bank **approve** the loan (1) or not (0)?

**Your task:** run it. Then, in the next cell, write a short comment **after every line**: what does this line do?

In [2]:
loans = pd.read_csv("loans.csv")
X = loans[["monthly_income", "loan_amount"]].values
y = loans[["approved"]].values

X = StandardScaler().fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

torch.manual_seed(42)
model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    guess = model(X)
    loss = loss_fn(guess, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("final loss:", round(loss.item(), 3))

final loss: 0.452


In [ ]:
# Part A code with an explanation after every line.
loans = pd.read_csv("loans.csv")  # read the loan table from the CSV file
X = loans[["monthly_income", "loan_amount"]].values  # select the two input features
y = loans[["approved"]].values  # select the binary target column
X = StandardScaler().fit_transform(X)  # standardize both input features
X = torch.tensor(X, dtype=torch.float32)  # convert inputs to float tensors
y = torch.tensor(y, dtype=torch.float32)  # convert binary labels to float tensors
torch.manual_seed(42)  # make random initialization reproducible
model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())  # build a binary classifier
loss_fn = nn.BCELoss()  # measure binary classification error
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # update weights with Adam
for epoch in range(100):  # train for 100 passes through the data
    guess = model(X)  # calculate predictions
    loss = loss_fn(guess, y)  # calculate prediction error
    optimizer.zero_grad()  # clear gradients from the previous step
    loss.backward()  # calculate gradients by backpropagation
    optimizer.step()  # update model parameters
print("final loss:", round(loss.item(), 3))  # display the final loss


**✏️ Questions about Part A** (write your answer in the cell below each one)

1. Why does this model end with `nn.Sigmoid()`?
2. Why do we use `BCELoss` here?
3. In Part B we predict **3** results. Which last layer and which loss will we use there?

*Your answers:*

1. `nn.Sigmoid()` converts the final value into a probability between 0 and 1 for binary approval.
2. `BCELoss` measures the error between predicted binary probabilities and the true 0/1 labels.
3. Use `nn.Linear(16, 3)` as the final layer and `nn.CrossEntropyLoss()`. Cross-entropy receives raw logits, so no softmax layer is needed.


---
# Part B: Mini-project: predict exam results

A college in Lahore wants to know early which students need help.
For **400 students** they have 5 numbers and the final **result**.

| Column | Meaning |
|---|---|
| `study_hours` | hours of study per day (0 to 6) |
| `attendance_pct` | attendance in % (40 to 100) |
| `assignments_done` | assignments done, out of 10 |
| `sleep_hours` | hours of sleep per night |
| `phone_hours` | hours on the phone per day |
| `result` | **0 = Fail, 1 = Pass, 2 = Distinction** |

**Goal:** train a network that guesses `result`. Then check it on the test students.

These 2 classes are the same as in class. **Run this cell, do not change it.**

In [ ]:
class Net(nn.Module):
    def __init__(self, n_inputs, n_hidden, n_outputs):
        super().__init__()                          
        self.hidden = nn.Linear(n_inputs, n_hidden) 
        self.relu = nn.ReLU()                       
        self.output = nn.Linear(n_hidden, n_outputs)

    def forward(self, x):
        return self.output(self.relu(self.hidden(x)))


class Trainer:
    def __init__(self, model, loss_fn, lr):
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.train_losses, self.val_losses = [], []

    def fit(self, X_train, y_train, X_val, y_val, epochs):
        for epoch in range(epochs):
            loss = self.loss_fn(self.model(X_train), y_train)   
            self.optimizer.zero_grad()                          
            loss.backward()                                     
            self.optimizer.step()                               
            self.train_losses.append(loss.item())
            with torch.no_grad():                               
                self.val_losses.append(self.loss_fn(self.model(X_val), y_val).item())

    def predict(self, X):
        with torch.no_grad():
            return self.model(X)

    def plot(self):
        plt.plot(self.train_losses, label="train (homework)")
        plt.plot(self.val_losses, label="validation (mock exam)")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.show()

## Step 1: Load and look

- Read `exam_results.csv` into `exams`.
- Print the first 5 rows and the shape.
- Count how many students are in each result with `.value_counts()`.

In [4]:
exams = pd.read_csv("exam_results.csv")
print(exams.head())
print("shape:", exams.shape)
print("students per result:")
print(exams["result"].value_counts())


   study_hours  attendance_pct  assignments_done  sleep_hours  phone_hours  \
0          3.8              60                 6          8.3          3.5   
1          5.4              96                10          8.4          2.0   
2          4.7              49                 6          7.1          5.7   
3          1.4              71                 1          6.6          5.8   
4          1.8              45                10          5.0          6.6   

   result  
0       1  
1       2  
2       1  
3       0  
4       0  
shape: (400, 6)
students per result:
result
1    180
0    140
2     80
Name: count, dtype: int64


## Step 2: X and y

- `X` = the 5 input columns (everything **except** `result`). Hint: `exams.drop(columns="result")`
- `y` = the `result` column.
- Use `.values` on both.

In [6]:
X = exams.drop(columns="result").values
y = exams["result"].values
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (400, 5)
y shape: (400,)


## Step 3: Split 60 / 20 / 20

Homework (train), mock exam (validation), final exam (test). Use `random_state=42` both times.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print(X_train.shape, X_val.shape, X_test.shape)


## Step 4: Scale

- `attendance_pct` is 40 to 100, `sleep_hours` is 4 to 9. Big numbers would push the weights too much.
- Make a `StandardScaler`. **Fit it on train only**, then transform train, val and test.
- Why train only? The test students are the "final exam". We must not peek at them.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)
print("train means:", X_train.mean(axis=0).round(3))


## Step 5: Tensors

- Inputs → `torch.float32` (decimals).
- Answers → `torch.long` (whole numbers), because `CrossEntropyLoss` wants the class number 0, 1, 2.

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)


## Step 6: Build and train

- Model: `Net` with **5 inputs, 16 hidden, 3 outputs**. Write `torch.manual_seed(42)` first.
- Loss: `nn.CrossEntropyLoss()`. Learning rate `0.01`.
- Train **100 epochs** with `Trainer`, then `.plot()` the losses.

In [ ]:
torch.manual_seed(42)
model = Net(n_inputs=5, n_hidden=16, n_outputs=3)
trainer = Trainer(model, nn.CrossEntropyLoss(), lr=0.01)
trainer.fit(X_train_t, y_train_t, X_val_t, y_val_t, epochs=100)
trainer.plot()


**✏️ Look at your plot:** do both lines go down? Does the validation line stop going down or go up at the end?

*Your answer:*

Both losses should generally decrease as the model learns. The validation loss may flatten or rise near the end if the model starts overfitting the training data.


## Step 7: Evaluate

- Guess for the **train** and **test** students: `trainer.predict(...)`, then `.argmax(dim=1)`.
- Accuracy = correct ÷ total. Print both.
- Make a table of real vs guessed with `pd.crosstab(y_test, test_guess)`.

In [ ]:
train_guess = trainer.predict(X_train_t).argmax(dim=1).numpy()
test_guess = trainer.predict(X_test_t).argmax(dim=1).numpy()
train_accuracy = (train_guess == y_train).mean()
test_accuracy = (test_guess == y_test).mean()
print(f"train accuracy: {train_accuracy:.1%}")
print(f"test accuracy: {test_accuracy:.1%}")
print(pd.crosstab(
    pd.Series(y_test, name="real"),
    pd.Series(test_guess, name="guessed")
).reindex(index=[0, 1, 2], columns=[0, 1, 2], fill_value=0))


**✏️ Questions:**
1. Is test accuracy lower than train accuracy? What does that mean?
2. In the table, which mistakes happen most? (Fail ↔ Pass, Pass ↔ Distinction, or Fail ↔ Distinction?)

*Your answers:*

1. Test accuracy (88.8%) is lower than train accuracy (96.2%) because the test students were not used to adjust the weights. The difference is evidence of some overfitting.
2. The most common mistake is Pass predicted as Fail (5 students); the next mistakes are Pass/Distinction in both directions (2 each). Fail/Distinction mistakes do not occur in this test split.


## Step 8: Predict for new students

| Name | study | attendance | assignments | sleep | phone |
|---|---|---|---|---|---|
| Ali | 5.0 | 90 | 9 | 7.5 | 1.0 |
| Sara | 1.0 | 55 | 2 | 5.0 | 6.0 |

- Put them in a table (2 rows × 5 numbers).
- **Scale with the same `scaler`** (only `.transform`).
- Tensor → `trainer.predict` → `torch.softmax(..., dim=1)` to see the chances.

In [ ]:
new_students = pd.DataFrame({
    "study_hours": [5.0, 1.0],
    "attendance_pct": [90, 55],
    "assignments_done": [9, 2],
    "sleep_hours": [7.5, 5.0],
    "phone_hours": [1.0, 6.0],
}, index=["Ali", "Sara"])
new_scaled = scaler.transform(new_students.values)
new_tensor = torch.tensor(new_scaled, dtype=torch.float32)
new_logits = trainer.predict(new_tensor)
new_probabilities = torch.softmax(new_logits, dim=1)
new_predictions = new_probabilities.argmax(dim=1).numpy()
results = new_students.copy()
results["predicted_result"] = new_predictions
results[["fail_probability", "pass_probability", "distinction_probability"]] = new_probabilities.numpy()
print(results)


---
# Part C: Experiments

One setting at a time. Run `run(...)` and **write the numbers in the table**.

In [ ]:
def run(n_hidden=16, lr=0.01, epochs=100):
    torch.manual_seed(42)
    model = Net(5, n_hidden, 3)
    trainer = Trainer(model, nn.CrossEntropyLoss(), lr=lr)
    trainer.fit(X_train_t, y_train_t, X_val_t, y_val_t, epochs=epochs)
    train_acc = (trainer.predict(X_train_t).argmax(dim=1).numpy() == y_train).mean()
    test_acc = (trainer.predict(X_test_t).argmax(dim=1).numpy() == y_test).mean()
    print(f"hidden={n_hidden}, lr={lr}, epochs={epochs} → train {train_acc:.1%}, test {test_acc:.1%}")

run()   # our model from Part B

In [ ]:
experiment_results = []
for label, kwargs in [
    ("normal (16, 0.01, 100)", {}),
    ("lr = 0.001", {"lr": 0.001}),
    ("lr = 0.1", {"lr": 0.1}),
    ("hidden = 4", {"n_hidden": 4}),
    ("hidden = 64", {"n_hidden": 64}),
    ("epochs = 20", {"epochs": 20}),
    ("epochs = 300", {"epochs": 300}),
]:
    torch.manual_seed(42)
    experiment_model = Net(5, kwargs.get("n_hidden", 16), 3)
    experiment_trainer = Trainer(
        experiment_model, nn.CrossEntropyLoss(), lr=kwargs.get("lr", 0.01)
    )
    experiment_trainer.fit(
        X_train_t, y_train_t, X_val_t, y_val_t, epochs=kwargs.get("epochs", 100)
    )
    experiment_train = (
        experiment_trainer.predict(X_train_t).argmax(dim=1).numpy() == y_train
    ).mean()
    experiment_test = (
        experiment_trainer.predict(X_test_t).argmax(dim=1).numpy() == y_test
    ).mean()
    experiment_results.append((label, experiment_train, experiment_test))
    print(f"{label}: train {experiment_train:.1%}, test {experiment_test:.1%}")


| Change | Train acc | Test acc | What happened? |
|---|---:|---:|---|
| normal (16, 0.01, 100) | 96.2% | 88.8% | Baseline. |
| lr = 0.001 | 75.4% | 72.5% | Smaller steps; learning is slower. |
| lr = 0.1 | 99.6% | 90.0% | Larger steps can overshoot, but this run still converged. |
| hidden = 4 | 91.2% | 91.2% | Smaller capacity; this run generalized well. |
| hidden = 64 | 99.6% | 90.0% | More capacity fits training data better. |
| epochs = 20 | 88.8% | 88.8% | Less training; the model has learned less. |
| epochs = 300 | 99.6% | 87.5% | More training improves train accuracy but slightly overfits. |


---
# Part D: Short questions

Answer in 1–2 lines each.

1. What does **one epoch** mean?
2. What do `loss.backward()` and `optimizer.step()` do?
3. Why do we split off **test** data and never train on it?
4. Why did we **scale** the columns?
5. Why is there **no softmax layer** in our `Net`?
6. Train 96%, test 82%. What is the name for this gap, and give one way to make it smaller.

*Your answers:*

1. One epoch is one complete pass through all training examples.
2. `loss.backward()` computes gradients for the model parameters; `optimizer.step()` uses those gradients to update the parameters.
3. Test data gives an unbiased final check on unseen data, so training must not use it.
4. Scaling puts features on comparable ranges so large-unit columns do not dominate learning.
5. `CrossEntropyLoss` applies the needed log-softmax internally and expects raw logits.
6. The gap is overfitting. Use early stopping, more data, regularization, or a smaller network.


---
# ? Checkpoint

- [x] I can explain what every line of the Part A code does.
- [x] I can load a CSV and make X and y.
- [x] I can split into train / validation / test and scale with train only.
- [x] I can make tensors (float32 for inputs, long for classes).
- [x] I can build a `Net`, train it with `Trainer`, and read the loss plot.
- [x] I can work out accuracy on test data.
- [x] I know when to use Sigmoid + BCELoss and when to use CrossEntropyLoss.
- [x] I can predict for a new person (scale first!).


---
# 🏠 Problem

Same steps, **yes/no** problem: will the bank approve the loan? Use `loans.csv`.

1. X = `monthly_income`, `loan_amount`, `years_in_job`, `has_defaulted_before`. y = `approved`.
2. Split 60 / 20 / 20, scale with train only.
3. Tensors: X **float32**, y **float32** with shape (n, 1). (BCELoss wants decimals.)
4. Model: `nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())`, loss `nn.BCELoss()`, 100 epochs.
5. Guess = chance > 0.5. Print the test accuracy.
6. Write 2 lines: what changed compared with the exam project, and why?

In [ ]:
loan_features = ["monthly_income", "loan_amount", "years_in_job", "has_defaulted_before"]
loan_data = pd.read_csv("loans.csv")
loan_X = loan_data[loan_features].values
loan_y = loan_data["approved"].values.reshape(-1, 1)
loan_X_train, loan_X_temp, loan_y_train, loan_y_temp = train_test_split(
    loan_X, loan_y, test_size=0.4, random_state=42, stratify=loan_y
)
loan_X_val, loan_X_test, loan_y_val, loan_y_test = train_test_split(
    loan_X_temp, loan_y_temp, test_size=0.5, random_state=42, stratify=loan_y_temp
)
loan_scaler = StandardScaler()
loan_X_train = loan_scaler.fit_transform(loan_X_train)
loan_X_val = loan_scaler.transform(loan_X_val)
loan_X_test = loan_scaler.transform(loan_X_test)
loan_X_train_t = torch.tensor(loan_X_train, dtype=torch.float32)
loan_X_val_t = torch.tensor(loan_X_val, dtype=torch.float32)
loan_X_test_t = torch.tensor(loan_X_test, dtype=torch.float32)
loan_y_train_t = torch.tensor(loan_y_train, dtype=torch.float32)
loan_y_val_t = torch.tensor(loan_y_val, dtype=torch.float32)
loan_model = nn.Sequential(
    nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid()
)
loan_loss_fn = nn.BCELoss()
loan_optimizer = torch.optim.Adam(loan_model.parameters(), lr=0.01)
for _ in range(100):
    loan_guess = loan_model(loan_X_train_t)
    loan_loss = loan_loss_fn(loan_guess, loan_y_train_t)
    loan_optimizer.zero_grad()
    loan_loss.backward()
    loan_optimizer.step()
loan_test_probability = loan_model(loan_X_test_t).detach()
loan_test_guess = (loan_test_probability > 0.5).float()
print(f"loan test accuracy: {(loan_test_guess == torch.tensor(loan_y_test)).float().mean().item():.1%}")

